# 第八节 ResNet-经典卷积神经网络

## 实验目标
通过本案例的学习：

1. 了解ResNet18的网络结构；
2. 掌握模型的保存和加载方法；
3. 掌握批量测试图片的方法；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0，需使用 <font color='red' >GPU</font> 运行，请查看[《ModelArts CodeLab介绍》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0010.html#section3)了解切换硬件规格的方法；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

## 案例内容介绍
ResNet系列网络是近几年来的经典图像分类网络，该系列网络有不同的层数，从低到高分别有resnet18、resnet34、resnet50、resnet101、resnet152等，本案例将使用renet18网络来实现手写数字识别。

### 1. 加载数据集
由于resnet18网络参数量比LeNet-5的要大，因此训练过程对显存就有更大的要求，我们可能无法将整个手写数字识别的6万个样本一次性加载进来进行训练，因此我们要分批次加载训练集进行训练。  
使用torch.utils.data.DataLoader工具可以很简单将数据集构造为一个数据生成器，每次只取出一小批的数据，实现代码如下，详情请查看代码注释。

In [1]:
import os
import torch
from torchvision import datasets, transforms

mnist_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081)),
])  # 由于resnet系列网络要求图片输入的尺寸是3*244*244，而MNIST数据集图片的尺寸是1*28*28，所以我们要构造一个图像变换操作，将MNIST的图片变换成3*244*244

train_batch_size = 64  # 训练集的批次大小
test_batch_size = 256  # 测试集的批次大小
torch.manual_seed(0)
datasets_dir = '../datasets'
if not os.path.exists(datasets_dir):
    os.makedirs(datasets_dir)
import moxing as mox
if not os.path.exists(os.path.join(datasets_dir, 'MNIST_data.zip')):
    mox.file.copy('obs://modelarts-labs-bj4/course/hwc_edu/deep_learning/datasets/MNIST_data.zip', 
                  os.path.join(datasets_dir, 'MNIST_data.zip'))
    os.system('cd %s; unzip MNIST_data.zip' % (datasets_dir))
    
train_dataset = datasets.MNIST(os.path.join(datasets_dir, 'MNIST_data'), train=True, download=True, transform=mnist_transform)  # 加载训练集
test_dataset = datasets.MNIST(os.path.join(datasets_dir, 'MNIST_data'), train=False, download=True, transform=mnist_transform)  # 加载测试集
train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=train_batch_size)  # 构造训练集批次生成器
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=True, batch_size=test_batch_size)  # 构造测试集批次生成器

INFO:root:Using MoXing-v1.17.3-

INFO:root:Using OBS-Python-SDK-3.20.7


### 2. 两行代码定义Resnet18网络
torchvision中定义了很多常用的网络结构，直接调用即可完成网络结构的定义。需要注意的一点就是，在调用torchvision的类进行了网络结构的定义后，要修改网络的最后一层全连接层，将该层的输出节点数改成实际分类任务的类别数，具体代码如下所示，详情请查看代码注释。

In [2]:
import torchvision
from torch import nn

In [3]:
net = torchvision.models.resnet18(pretrained=True)  # 一行代码实现resnet18的网络结构定义，pretrained=True表示将加载resnet18的官方预训练参数文件
net.fc = nn.Linear(net.fc.in_features, out_features=10)  # net.fc就是指模型的最后一层全连接层，net.fc.in_features是指该层的输入节点数，out_features是指该层的输出节点数，也就是要分类的类别数，MNIST手写数字识别是十分类任务，因此填写为10

### 3. 定义评价函数

由于网络结构是使用封装好的类，因此我们需要在类外单独定义评价函数  
由于训练集和测试集都是分批次进行预测，所以我们要先统计每个批次中预测正确的样本，最后等所有批次统计完成后才能计算准确率

In [4]:
def evaluate(pred_y, true_y):
    # pred_labels = torch.argmax(pred_y, dim=1)
    # acc = (pred_labels == true_y).float().mean()
    pred_labels = pred_y.argmax(dim=1, keepdim=True)
    correct_num = pred_labels.eq(true_y.view_as(pred_labels)).sum().item()
    return correct_num

### 4. 交叉熵损失函数
与上一节代码一致

In [5]:
import torch.nn.functional as F
loss_fun = F.cross_entropy

### 5. 实现GPU训练的梯度下降算法
与上一节代码基本一致

In [6]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 如果当前机器的cuda可用，则用GPU进行训练
net = net.to(device)  # 将模型拷贝到GPU上
optimizer = optim.SGD(net.parameters(), lr=0.01)

### 6. 实现训练函数

In [7]:
def train(net, train_loader, test_loader, max_epochs=100):
    import gc
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(1, max_epochs + 1):
        net.train()  # 切换为训练模式
        train_loss = 0.0
        train_correct_num = 0
        for iter_idx, (train_x_batch, train_y_batch) in enumerate(train_loader):
            train_x_batch, train_y_batch = train_x_batch.to(device), train_y_batch.to(device)
            pred_y_train_batch = net.forward(train_x_batch)  # 前向传播            
            current_train_loss = loss_fun(pred_y_train_batch, train_y_batch)  # 计算损失
            train_loss += current_train_loss.item()
            train_correct_num += evaluate(pred_y_train_batch, train_y_batch)

            # 计算梯度，更新权值
            current_train_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            if iter_idx % 50 == 0:
                print('epoch: %d, iter_idx: %d, train_loss: %.4f' % (epoch, iter_idx, current_train_loss.item()))
            
        train_loss /= len(train_loader.dataset)
        train_acc = float(train_correct_num) / len(train_loader.dataset)
        

        if (epoch == 1) or (epoch % 200 == 0):
            net.eval()  # 切换为评价模式，评价模式不计算梯度，计算更快
            test_loss = 0.0
            test_correct_num = 0.0
            for iter_idx, (test_x_batch, test_y_batch) in enumerate(test_loader):
                test_x_batch, test_y_batch = test_x_batch.to(device), test_y_batch.to(device)
                pred_y_test_batch = net.forward(test_x_batch)
                test_loss += loss_fun(pred_y_test_batch, test_y_batch).item()
                test_correct_num += evaluate(pred_y_test_batch, test_y_batch)
            
            test_loss /= len(test_loader.dataset)
            test_acc = float(test_correct_num) / len(test_loader.dataset)
            print('epoch %d, train_loss %.4f, test_loss %.4f, train_acc: %.4f, test_acc: %.4f' % (epoch, train_loss, test_loss, train_acc, test_acc))
    return train_losses, test_losses, train_accs, test_accs

### 7. 开始训练
代码基本与上一节一致，只是将max_epochs改成了1  
训练耗时约150秒

In [8]:
import time
start_time = time.time()
max_epochs = 1
train_losses, test_losses, train_accs, test_accs = train(net, train_loader, test_loader, max_epochs=max_epochs)
print('cost time: %.1f s' % int(time.time() - start_time))

epoch: 1, iter_idx: 0, train_loss: 2.5492

epoch: 1, iter_idx: 50, train_loss: 0.3100

epoch: 1, iter_idx: 100, train_loss: 0.1354

epoch: 1, iter_idx: 150, train_loss: 0.0844

epoch: 1, iter_idx: 200, train_loss: 0.0898

epoch: 1, iter_idx: 250, train_loss: 0.1829

epoch: 1, iter_idx: 300, train_loss: 0.0355

epoch: 1, iter_idx: 350, train_loss: 0.0869

epoch: 1, iter_idx: 400, train_loss: 0.0602

epoch: 1, iter_idx: 450, train_loss: 0.1429

epoch: 1, iter_idx: 500, train_loss: 0.0446

epoch: 1, iter_idx: 550, train_loss: 0.0372

epoch: 1, iter_idx: 600, train_loss: 0.0298

epoch: 1, iter_idx: 650, train_loss: 0.0249

epoch: 1, iter_idx: 700, train_loss: 0.0210

epoch: 1, iter_idx: 750, train_loss: 0.0231

epoch: 1, iter_idx: 800, train_loss: 0.0131

epoch: 1, iter_idx: 850, train_loss: 0.0278

epoch: 1, iter_idx: 900, train_loss: 0.0202

epoch 1, train_loss 0.0019, test_loss 0.0001, train_acc: 0.9745, test_acc: 0.9916

cost time: 157.0 s


从上面的输出结果，可以看到ResNet18模型仅训练一个epoch，耗时仅150秒左右，就在手写数字识别任务的测试集上达到了0.99以上的准确率。  
该准确率可以达到应用水平，下面将保存模型、并加载模型进行批量图片预测，看看真实的预测效果如何。

### 8. 保存模型

In [9]:
torch.save(net, './resnet18_mnist.pth')

### 9. 加载模型

In [10]:
import torch 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet18_mnist = torch.load('./resnet18_mnist.pth', map_location=device)

### 10. 进行批量图片预测

模型训练时，对训练图片进行了图像转换操作，加载图片进行测试时，也需要同样的转换操作，下面的代码直接本案例开头第一步中的代码

In [11]:
from torchvision import transforms

mnist_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081)),
])  # 由于resnet系列网络要求图片输入的尺寸是3*244*244，而MNIST数据集图片的尺寸是1*28*28，所以我们要构造一个图像变换操作，将MNIST的图片变换成3*244*244

In [12]:
import os
import numpy as np
from glob import glob
from PIL import Image

datasets_dir = '../datasets'
test_img_dir = os.path.join(datasets_dir, 'MNIST_data/test_imgs')
files = glob(os.path.join(test_img_dir, '*.jpg'))

resnet18_mnist.eval()  # 将模型转换为评价模式
pred_labels = []
show_img = np.zeros((28, 1), dtype= np.uint8)
for file_path in files:
    src_img = Image.open(file_path)  # 加载单张图片
    img = mnist_transform(src_img)   # 对图片进行图像转换
    img = img.unsqueeze(0)
    img = img.to(device)
    pred_y = resnet18_mnist.forward(img)
    pred_label = torch.argmax(pred_y)
    pred_labels.append(pred_label.item())
    show_img = np.hstack((show_img, np.array(src_img)))

print(pred_labels)  # 打印预测结果
Image.fromarray(show_img)  # 显示预测图片

[4, 6, 9, 0, 2, 9, 9, 4, 0, 6, 6, 8, 2, 8, 1, 9, 1, 1, 5, 3]


从上面的结果可以看到，20张图片中，仅一张手写不太规范的数字“9”被识别成了“4”，其他图片全部预测正确，说明真实的预测效果是还不错的。

至此，本案例完成。